# DiCo-NLI Track 1 — Assignment 2 experiments (Colab)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Run the cells in order. Everything (code, data, official scorer) comes from the GitHub clone in cell 1.

Models are trained in **fp32** (DeBERTa-v3 gives NaN in fp16 and on Apple MPS).

In [ ]:
# 1. Setup: clone the repo (data, official scorer, src/data.py)
!git clone -q https://github.com/MuneebBaig71/DiCo-NLI_SemEval.git
%cd DiCo-NLI_SemEval
!pip -q install sentencepiece
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Check the A2 files came with the clone
from pathlib import Path
from google.colab import files
need = ["src/decoding.py", "src/train_a2.py", "06_decoding_experiments.py"]
missing = [p for p in need if not Path(p).exists()]
print("MISSING:", missing if missing else "none - all A2 files present")

In [ ]:
# 3. (Recommended) save results to Google Drive so a disconnect doesn't lose them
from google.colab import drive
drive.mount("/content/drive")
DRIVE_OUT = "/content/drive/MyDrive/DiCo_A2_results"

In [ ]:
# 4. Smoke test (~2 min): 1 epoch of DeBERTa-v3 on 300 rows, with the consistency loss on.
#    Checks: no NaN, the loss terms print, files get written. Not a real result.
import importlib, src.decoding, src.train_a2
importlib.reload(src.decoding); importlib.reload(src.train_a2)
from src.train_a2 import run
meta = run(model="microsoft/deberta-v3-base", epochs=1, max_train_rows=300,
           lambda_cons=1.0, tag="smoke", out_dir="results_smoke")
print(meta["history"])

In [ ]:
# 5. Main runs, seed 42 (about 10 min each on a T4).
#    M0 = A1 RoBERTa rerun under the new protocol (fair comparison)
#    M1 = plain DeBERTa-v3, M2 = + consistency loss, M3 = M2 + Neg weighting
import json, shutil, pandas as pd
from pathlib import Path

EXPS = [
    dict(tag="M0_roberta",        model="roberta-base"),
    dict(tag="M1_deberta",        model="microsoft/deberta-v3-base"),
    dict(tag="M2_deberta_cons",   model="microsoft/deberta-v3-base", lambda_cons=1.0),
    dict(tag="M3_deberta_cons_neg", model="microsoft/deberta-v3-base", lambda_cons=1.0, neg_weight=2.0),
]
SEED = 42

for e in EXPS:
    tag = f"{e['tag']}_seed{SEED}"
    if Path(f"results_a2/runs/{tag}.json").exists():
        print("skip (done):", tag); continue
    run(**{**e, "tag": tag, "seed": SEED})
    try:   # copy after every run, so a disconnect loses at most one run
        shutil.copytree("results_a2", DRIVE_OUT, dirs_exist_ok=True)
    except NameError:
        pass

In [ ]:
# 6. Score every run on dev with the official scorer, under all 4 decoding rules
!python 06_decoding_experiments.py

In [ ]:
# 7. Training summary (best epoch chosen on the validation split, not dev)
rows = []
for f in sorted(Path("results_a2/runs").glob("*.json")):
    m = json.loads(f.read_text())
    rows.append(dict(tag=m["tag"], best_epoch=m["best_epoch"],
                     best_val_wf1=round(m["best_val_wf1"], 4), minutes=round(m["seconds"] / 60, 1)))
pd.DataFrame(rows)

In [ ]:
# 8. Download everything as a zip (unzip into your Mac folder)
!zip -qr results_a2.zip results_a2
files.download("results_a2.zip")

In [ ]:
# 10. Checkpoint save/load check (~5 min, A2 brief section 8).
#     Fresh runtime is fine: run cells 1 and 3 first, then this cell.
#     Trains M3 seed 42 once with save_checkpoint=True, reloads the saved model from disk,
#     predicts dev again, and checks the reloaded predictions match the trained model's.
import importlib, json, shutil, numpy as np, pandas as pd, torch, transformers
from pathlib import Path
import src.train_a2; importlib.reload(src.train_a2)
from src.train_a2 import run, predict_from_checkpoint
from src.data import LABELS, split_path, score, headline

TAG, OUT = "M3_ckpt_check_seed42", "results_ckpt"
meta = run(model="microsoft/deberta-v3-base", lambda_cons=1.0, neg_weight=2.0, seed=42,
           tag=TAG, out_dir=OUT, save_checkpoint=True)
ckpt = f"{OUT}/checkpoints/{TAG}"

saved = pd.read_csv(f"{OUT}/predictions/{TAG}_dev_probs.csv").set_index("instance_id")
ids, reloaded = predict_from_checkpoint(ckpt, "dev")
diff = np.abs(saved.loc[ids, LABELS].to_numpy() - reloaded).max()
same_labels = (saved.loc[ids, LABELS].to_numpy().argmax(1) == reloaded.argmax(1)).mean()

pd.DataFrame({"instance_id": ids, "label": [LABELS[k] for k in reloaded.argmax(1)]}
             ).to_csv(f"{OUT}/reloaded_dev.csv", index=False)
print("best epoch:", meta["best_epoch"], "| train minutes:", round(meta["seconds"] / 60, 1))
print("checkpoint files:", sorted(p.name for p in Path(ckpt).iterdir()))
print("checkpoint size (MB):", round(sum(p.stat().st_size for p in Path(ckpt).iterdir()) / 1e6))
print("max |prob diff| trained vs reloaded:", diff)
print("identical argmax labels:", same_labels)
print("reloaded model, argmax, official scorer:", headline(score(f"{OUT}/reloaded_dev.csv", "dev")))
print("versions: torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPU", torch.cuda.get_device_name(0))

# compare with the original M3 seed-42 run (same seed and config, separate training run)
orig = pd.read_csv(f"/content/drive/MyDrive/DiCo_A2_results/predictions/M3_deberta_cons_neg_seed42_dev_probs.csv").set_index("instance_id")
print("max |prob diff| vs original seed-42 run:", np.abs(orig.loc[ids, LABELS].to_numpy() - reloaded).max(),
      "| label agreement:", (orig.loc[ids, LABELS].to_numpy().argmax(1) == reloaded.argmax(1)).mean())

shutil.copytree(OUT, "/content/drive/MyDrive/DiCo_A2_ckpt_check", dirs_exist_ok=True)
print("copied to Drive: MyDrive/DiCo_A2_ckpt_check")

## English-track experiments (outside Assignment 2)

Everything below writes to `results_exp/` and never touches `results_a2/`, so the Assignment 2 numbers stay frozen.
Fresh runtime: run cells 1 and 3 first, then the two cells below.

In [ ]:
# 11. English experiments: stronger starting points for the same pipeline.
#     E1 = DeBERTa-v3-base already fine-tuned on NLI data (MNLI/FEVER/ANLI), new 4-class head
#     E2 = DeBERTa-v3-large
#     E3 = DeBERTa-v3-large already fine-tuned on NLI data
#     All use the M3 recipe (consistency loss + Neg weight). Large models use lr 1e-5.
#     Time on a T4: about 3.5 min per base run, about 10-12 min per large run.
import importlib, json, shutil
from pathlib import Path
import src.train_a2; importlib.reload(src.train_a2)
from src.train_a2 import run

OUT, DRIVE_EXP = "results_exp", "/content/drive/MyDrive/DiCo_EXP_results"
if Path(DRIVE_EXP).exists():   # resume after a disconnect
    shutil.copytree(DRIVE_EXP, OUT, dirs_exist_ok=True)

M3 = dict(lambda_cons=1.0, neg_weight=2.0, out_dir=OUT)
EXPS = [
    dict(tag="E1_nli_base",  model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli", reinit_head=True, **M3),
    dict(tag="E2_large",     model="microsoft/deberta-v3-large", lr=1e-5, **M3),
    dict(tag="E3_nli_large", model="MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
         reinit_head=True, lr=1e-5, **M3),
]
SEEDS = [13, 42, 87]   # start with [42] if you only want a quick look

for e in EXPS:
    for seed in SEEDS:
        tag = f"{e['tag']}_seed{seed}"
        if Path(f"{OUT}/runs/{tag}.json").exists():
            print("skip (done):", tag); continue
        try:
            run(**{**e, "tag": tag, "seed": seed})
        except Exception as err:      # one failed run should not stop the rest
            print(f"FAILED {tag}: {type(err).__name__}: {err}"); continue
        shutil.copytree(OUT, DRIVE_EXP, dirs_exist_ok=True)
print("done")

In [ ]:
# 12. Compare the experiments with the Assignment 2 systems (validation and dev, plus ensembles).
#     Bring the A2 runs back from Drive so they appear in the same table.
import shutil
from pathlib import Path
if not list(Path("results_a2/predictions").glob("M3_*_val_probs.csv")):
    shutil.copytree("/content/drive/MyDrive/DiCo_A2_results", "results_a2", dirs_exist_ok=True)
!python 08_exp_summary.py results_exp results_a2
shutil.copytree("results_exp", "/content/drive/MyDrive/DiCo_EXP_results", dirs_exist_ok=True)